# S11 — Phase 2 computational-feasibility audit

This notebook reconstructs the trainable-parameter count of the frozen-backbone Phase 2 design and verifies the recorded run configurations. It does not claim a complete VRAM benchmark or comparative computational superiority.

In [ ]:
from pathlib import Path
import json
import platform
import time

import numpy as np
import pandas as pd
import torch
from torch import nn
from IPython.display import display

PROJECT = Path(r"C:\Users\Dell\Desktop\Publication_Clarck\Natural_Sampling")
BUNDLE_MODELS = (PROJECT / "Notebooks" / "Notebook Officiel" /
                 "Reviewer_Reproducibility_Bundle_20260823" / "03_models")
SITES = ("Ifran", "Maamoura", "Agadir")

assert PROJECT.is_dir(), PROJECT
assert BUNDLE_MODELS.is_dir(), BUNDLE_MODELS
print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
rows = []
for site in SITES:
    run_root = BUNDLE_MODELS / site / "Phase2" / "run_metadata"
    config_path = run_root / "config.json"
    done_path = run_root / "TRAIN_DONE.json"
    assert config_path.is_file(), config_path
    assert done_path.is_file(), done_path

    config = json.loads(config_path.read_text(encoding="utf-8"))
    done = json.loads(done_path.read_text(encoding="utf-8"))
    assert config["reference_frozen"] is True
    assert config["train_scope"] == "full official-style Conv3D temporal prediction head"

    rows.append({
        "landscape": site,
        "phase1_frozen": config["reference_frozen"],
        "trainable_parameters_recorded": int(config["trainable_parameters"]),
        "optimizer": config["optimizer"],
        "max_steps": int(config["max_steps"]),
        "elapsed_minutes": float(done["elapsed_seconds"]) / 60.0,
    })

run_evidence = pd.DataFrame(rows)
assert run_evidence["trainable_parameters_recorded"].nunique() == 1
display(run_evidence.round({"elapsed_minutes": 2}))


In [ ]:
ADAPTER_SOURCE = PROJECT / "Source" / "Training" / "growthloss_v6" / "src" / "b4_echosat_adapter.py"
assert ADAPTER_SOURCE.is_file(), ADAPTER_SOURCE
adapter_text = ADAPTER_SOURCE.read_text(encoding="utf-8")
for required_token in ("nn.Conv3d", "nn.GroupNorm(4, channels)", "padding_mode=\"replicate\"", "nn.Conv3d(channels, 1"):
    assert required_token in adapter_text, f"Missing architecture token: {required_token}"

def build_phase2_residual_head(channels: int = 64) -> nn.Sequential:
    return nn.Sequential(
        nn.Conv3d(channels, channels, kernel_size=3, padding=1, padding_mode="replicate"),
        nn.GroupNorm(4, channels),
        nn.ReLU(inplace=False),
        nn.Conv3d(channels, channels, kernel_size=3, padding=1, padding_mode="replicate"),
        nn.GroupNorm(4, channels),
        nn.ReLU(inplace=False),
        nn.Conv3d(channels, 1, kernel_size=1),
    )

head = build_phase2_residual_head()
parameter_rows = []
for name, parameter in head.named_parameters():
    parameter_rows.append({
        "tensor": name,
        "shape": str(tuple(parameter.shape)),
        "parameters": parameter.numel(),
    })
parameter_table = pd.DataFrame(parameter_rows)
computed_trainable = sum(p.numel() for p in head.parameters() if p.requires_grad)
recorded_trainable = int(run_evidence["trainable_parameters_recorded"].iloc[0])
assert computed_trainable == recorded_trainable == 221_633
display(parameter_table)
print(f"PASS: exact trainable parameter count = {computed_trainable:,}")


In [ ]:
bytes_per_fp32 = 4
weight_bytes = computed_trainable * bytes_per_fp32
# FP32 weights + gradients + two AdamW moment tensors.
adamw_state_bytes = weight_bytes * 4
memory_table = pd.DataFrame([
    {"quantity": "FP32 trainable weights", "MiB": weight_bytes / 2**20},
    {"quantity": "Weights + gradients + AdamW moments", "MiB": adamw_state_bytes / 2**20},
])
display(memory_table.round(3))


In [ ]:
RUN_GPU_MICROBENCHMARK = True
WARMUP_STEPS = 10
MEASURED_STEPS = 50
INPUT_SHAPE = (1, 64, 4, 96, 96)

if RUN_GPU_MICROBENCHMARK:
    assert torch.cuda.is_available(), "CUDA is required for this optional cell."
    device = torch.device("cuda:0")
    bench_head = build_phase2_residual_head().to(device).train()
    optimizer = torch.optim.AdamW(bench_head.parameters(), lr=1e-4, weight_decay=0.005)
    x = torch.randn(INPUT_SHAPE, device=device)

    def one_step():
        optimizer.zero_grad(set_to_none=True)
        y = bench_head(x)
        loss = y.square().mean()
        loss.backward()
        optimizer.step()

    for _ in range(WARMUP_STEPS):
        one_step()
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats(device)
    start = time.perf_counter()
    for _ in range(MEASURED_STEPS):
        one_step()
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - start
    microbenchmark = pd.DataFrame([{
        "gpu": torch.cuda.get_device_name(device),
        "input_shape": str(INPUT_SHAPE),
        "mean_step_ms_head_only": 1000 * elapsed / MEASURED_STEPS,
        "peak_allocated_MiB_head_only": torch.cuda.max_memory_allocated(device) / 2**20,
        "peak_reserved_MiB_head_only": torch.cuda.max_memory_reserved(device) / 2**20,
    }])
    display(microbenchmark.round(2))
else:
    print("GPU microbenchmark skipped. Set RUN_GPU_MICROBENCHMARK=True to execute it.")
